In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display

from tqdm import tqdm

# cross-sectional
# loader_cross_sectional_tb = features_loader_cross_sectional_tissue_based.FeaturesLoader
# loader_cross_sectional_wbb = features_loader_cross_sectional_whole_brain_based.FeaturesLoader

# logitudianl
# loader_logitudinal_wbb = features_loader_longitudinal_whole_brain_based.FeaturesLoader
# loader_logitudinal_tb = features_loader_longitudinal_tissue_based.FeaturesLoader

#registration
# loader_registration_wbb = features_loader_registration_whole_brain_based.FeaturesLoader
# loader_registration_tbb = features_loader_registration_tissue_based.FeaturesLoader

# approaches
# loader_approaches_wbb = features_loader_approaches_whole_brain_based.FeaturesLoader
# loader_approaches_tbb = features_loader_approaches_tissue_based.FeaturesLoader

np.set_printoptions(precision=4)

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from brainpred import paths, splits
from brainpred.utils import display as vis
from brainpred.utils import intensity, metrics, ml as uml
from brainpred.features.cross_sectional import tissue_based as features_loader_cross_sectional_tissue_based
from brainpred.features.cross_sectional import whole_brain as features_loader_cross_sectional_whole_brain_based
from brainpred.features.longitudinal import whole_brain_based as features_loader_longitudinal_whole_brain_based
from brainpred.features.longitudinal import tissue_based as features_loader_longitudinal_tissue_based
from brainpred.features.registration import whole_brain_based as features_loader_registration_whole_brain_based
from brainpred.features.registration import tissue_based as features_loader_registration_tissue_based
from brainpred.features.approaches import whole_brain_based as features_loader_approaches_whole_brain_based
from brainpred.features.approaches import tissue_based as features_loader_approaches_tissue_based
from brainpred.features.approaches.common import get_diffusion_model_ids

def get_save_image_path_name(img_name):
    return f"{paths.get_figures_folder()}/{img_name}.png"

# folder for the tables written by this notebook (patient-level data: never commit them)
current_path = paths.get_work_folder("analysis")

# Important regions of the brain

- **lateral ventricles** (R:43, L:4)
- **hipocampus** (R:53, L:17)
- **thalamus** (R:49, L:10)
- **cortex** (R:42, L:3)
    - entorhinal cortex (R:2006, L:1006)
    - inferior temporal gyri (R:2009, L:1009)
    - superior parietal (R:2029, L:1029)
    - middle frontal gyrus (R:2027, L:1027)
    - superior frontal gyrus (R:2028, L:1028)
    - precuneus (R:2025, L:1025)
    - cuneus (R:2005, L:1005)

**Other founds:**
- 3rd ventricle (14)

In [ ]:
# lat. ventricles
# hippocampus
# thalamus
# cortex
# entorhinal cortex
# inf. temporal. gyri
# sup. parietal
# mid. frontal gyrus
# sup. frontal gyrus
# precuneus
# cuneus
# 43_4_dice	53_17_dice	49_10_dice  42_3_dice	2006_1006_dice	2009_1009_dice	2029_1029_dice	2027_1027_dice	2028_1028_dice	2025_1025_dice	2005_1005_dice

ddc = {"lat. ventricles": "43_4_...", "hippocampus": "53_17_...", "thalamus": "49_10_...", "cortex": "42_3_...", "entorhinal cortex": "2006_1006_...", "inf. temporal. gyri": "2009_1009_...", "sup. parietal": "2029_1029_...", "mid. frontal gyrus": "2027_1027_...", "sup. frontal gyrus": "2028_1028_...", "precuneus": "2025_1025_...", "cuneus": "2005_1005_..."}

In [ ]:
val_images = splits.get_validation_ids()
nb_tissues = 96
# IMPORTANT_TISSUES_36_96 = [43, 4, 53, 17, 49, 10, 42, 3, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
if nb_tissues == 96:
    IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
    join_mean = [(43, 4), (53, 17), (49, 10), (2006, 1006), (2009, 1009), (2029, 1029), (2027, 1027), (2028, 1028), (2025, 1025), (2005, 1005)]
elif nb_tissues == 36:
    IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 42, 3, 14]
    join_mean = [(43, 4), (53, 17), (49, 10), (42, 3)]
elif nb_tissues == 3:
    join_mean = [(1, 1), (2, 2), (3, 3)]
    IMPORTANT_TISSUES = [1,2,3]


# Load features

In [ ]:
# ---------------- cross-sectional:
# ------------ 1. tissue-based
data_cross_sectional_wbb = features_loader_cross_sectional_whole_brain_based.FeaturesLoader().get_df(id_patients=val_images, remove_patients=False)
data_cross_sectional_tb = features_loader_cross_sectional_tissue_based.FeaturesLoader(nb_tissues, val_images).get_df()

# display(data_cross_sectional_tb.head())

# ---------------- longitudinal:
data_logitudinal_wbb = features_loader_longitudinal_whole_brain_based.FeaturesLoader().get_df(id_patients=val_images, remove_patients=False)
data_logitudinal_tb = features_loader_longitudinal_tissue_based.FeaturesLoader(nb_tissues, val_images).get_df()

# ---------------- registration:
data_registration_wbb = features_loader_registration_whole_brain_based.FeaturesLoader().get_df(id_patients=val_images, remove_patients=False)
data_registration_tb = features_loader_registration_tissue_based.FeaturesLoader(nb_tissues, val_images).get_df()

# ---------------- approaches:
data_approaches_wbb = {}
data_approaches_tb = {}

# dfield_similar_patches
data_approaches_wbb["dfield_similar_patches"] = features_loader_approaches_whole_brain_based.FeaturesLoader("dfield_similar_patches").get_df()
data_approaches_tb["dfield_similar_patches"] = features_loader_approaches_tissue_based.FeaturesLoader("dfield_similar_patches", nb_tissues, val_images).get_df()

data_approaches_wbb["MPGAN_paper_16_seg_loss"] = features_loader_approaches_whole_brain_based.FeaturesLoader("MPGAN_paper_16_seg_loss").get_df()
data_approaches_tb["MPGAN_paper_16_seg_loss"] = features_loader_approaches_tissue_based.FeaturesLoader("MPGAN_paper_16_seg_loss", nb_tissues, val_images).get_df()

data_approaches_wbb["MPGAN_paper_16"] = features_loader_approaches_whole_brain_based.FeaturesLoader("MPGAN_paper_16").get_df()
data_approaches_tb["MPGAN_paper_16"] = features_loader_approaches_tissue_based.FeaturesLoader("MPGAN_paper_16", nb_tissues, val_images).get_df()

data_approaches_wbb["MGAN_paper_16"] = features_loader_approaches_whole_brain_based.FeaturesLoader("MGAN_paper_16").get_df()
data_approaches_tb["MGAN_paper_16"] = features_loader_approaches_tissue_based.FeaturesLoader("MGAN_paper_16", nb_tissues, val_images).get_df()

data_approaches_wbb["StarGAN_V2_sagital"] = features_loader_approaches_whole_brain_based.FeaturesLoader("StarGAN_V2_sagital").get_df()
data_approaches_tb["StarGAN_V2_sagital"] = features_loader_approaches_tissue_based.FeaturesLoader("StarGAN_V2_sagital", nb_tissues, val_images).get_df()

data_approaches_wbb["diffusion_model"] = features_loader_approaches_whole_brain_based.FeaturesLoader("diffusion_model").get_df()
data_approaches_tb["diffusion_model"] = features_loader_approaches_tissue_based.FeaturesLoader("diffusion_model", nb_tissues, get_diffusion_model_ids()).get_df()


# display(data_approaches_wbb["dfield_similar_patches"].head(10))
# display(data_logitudinal_wbb.head(10))


# Find best patient

# t-score

In [ ]:
t_stat_A, p_val_A = stats.ttest_rel(data_approaches_wbb["dfield_similar_patches"]["ssim_hunt4"].values, data_logitudinal_wbb["ssim"].values)
t_stat_B, p_val_B = stats.ttest_rel(data_approaches_wbb["MPGAN_paper_16_seg_loss"]["ssim_hunt4"].values, data_logitudinal_wbb["ssim"].values)
t_stat_C, p_val_C = stats.ttest_rel(data_approaches_wbb["MGAN_paper_16"]["ssim_hunt4"].values, data_logitudinal_wbb["ssim"].values)
t_stat_D, p_val_D = stats.ttest_rel(data_approaches_wbb["StarGAN_V2_sagital"]["ssim_hunt4"].values, data_logitudinal_wbb["ssim"].values)


print(f"Metodo A: t = {t_stat_A}, p = {p_val_A}")
print(f"Metodo B: t = {t_stat_B}, p = {p_val_B}")
print(f"Metodo C: t = {t_stat_C}, p = {p_val_C}")
print(f"Metodo D: t = {t_stat_D}, p = {p_val_D}")

# Comparación de resultados
if p_val_A < 0.05:
    print("Método A es significativo.")
if p_val_B < 0.05:
    print("Método B es significativo.")
if p_val_C < 0.05:
    print("Método C es significativo.")
if p_val_D < 0.05:
    print("Método D es significativo.")

In [ ]:
def compute_mean_emisphres(data, join_mean, sufix="dice", desired_sufix=None):
    if desired_sufix is None:
        desired_sufix = sufix
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]
    _data = pd.DataFrame()
    for i, (t1, t2) in enumerate(join_mean):
        _data[columns[i]] = (data[f"{t1}_{sufix}"] + data[f"{t2}_{sufix}"]) / 2
    return _data


def compute_sum_emisphres(data, join_mean, sufix="vol_hunt3", desired_sufix=None):
    if desired_sufix is None:
        desired_sufix = sufix
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]
    _data = pd.DataFrame()
    for i, (t1, t2) in enumerate(join_mean):
        _data[columns[i]] =  (data[f"{t1}_{sufix}"] + data[f"{t2}_{sufix}"])
    if join_mean[0][0] == join_mean[0][1]: # just for 3 tissues segmentation
        return _data/2
    return _data


from sklearn.preprocessing import MinMaxScaler

def create_statistic_table(data, initial_data, columns, mult=1, std=False):
    if std:
        scaler = MinMaxScaler()
        data = pd.DataFrame(scaler.fit_transform(data[columns]), columns=columns).copy()
        initial_data = pd.DataFrame(scaler.fit_transform(initial_data[columns]), columns=columns).copy()


    res_dict = {}
    res_means_dict = {}
    res_sem_dict = {}
    res_t_test_dict = {}
    res_p_value_dict = {}
    for i, column in enumerate(columns):
        _initial_res = initial_data[column].values * mult
        res_dict[column] = data[column].values * mult

        t_stat_A, p_val_A = stats.ttest_rel(res_dict[column], _initial_res)
        res_t_test_dict[column] = t_stat_A
        res_p_value_dict[column] = p_val_A
        res_means_dict[column] = res_dict[column].mean()
        # res_sem_dict[column] = stats.sem(res_dict[column], axis=0)
        res_sem_dict[column] = res_dict[column].std()

    # create np table
    res_table = np.zeros((len(columns), 4))
    for i, column in enumerate(columns):
        res_table[i, 0] = res_means_dict[column]
        res_table[i, 1] = res_sem_dict[column]
        res_table[i, 2] = res_t_test_dict[column]
        res_table[i, 3] = res_p_value_dict[column]

    # create pd table
    df_mean = pd.DataFrame(res_table[:,0][:, np.newaxis].T, columns=columns)
    df_sem = pd.DataFrame(res_table[:,1][:, np.newaxis].T, columns=columns)
    df_t_test = pd.DataFrame(res_table[:,2][:, np.newaxis].T, columns=columns)
    df_p_value = pd.DataFrame(res_table[:,3][:, np.newaxis].T, columns=columns)
    return df_mean, df_sem, df_t_test, df_p_value






# Image similarity

In [ ]:

def display_dice_table(hunt=4):
    approaches = ["initial", "gt_registration", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]
    columns = ["ssim", "ncc", "psnr"]

    mean_table = np.zeros((len(approaches), len(columns)))
    sem_table = np.zeros((len(approaches), len(columns)))
    t_test_table = np.zeros((len(approaches), len(columns)))
    p_value_table = np.zeros((len(approaches), len(columns)))

    columns_rename = dict([(f"{col}_hunt{hunt}",col) for col in columns])
    initial_data = data_logitudinal_wbb[columns]
    
    for i, approach in enumerate(approaches):
        if approach in [None, "initial"]:
            data = initial_data
        elif approach == "gt_registration":
            data = data_registration_wbb[columns]
        else:
            # data = compute_mean_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix)
            # data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix) / 2
            data = data_approaches_wbb[approach].copy()
            data.rename(columns=columns_rename, inplace=True)
            data = data[columns]
        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=1)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.3f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_dice_table()
df_mean_hunt3, df_sem_hunt3, df_t_test_hunt3, df_p_value_hunt3 = display_dice_table(hunt=3)
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean) 
# print("means HUNT3")
# display(df_mean_hunt3)
print("std error")
display(df_sem)
# print("t-test")
# display(df_t_test)
# print("p-value")
# display(df_p_value)


# Dice score

In [ ]:

def display_dice_table(hunt=4):
    # approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "StarGAN_V2_sagital"]
    # approaches = ["initial", "gt_registration", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]
    # approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "gt_registration"]
    approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="dice"
    initial_data = compute_mean_emisphres(data_logitudinal_tb, join_mean, sufix="dice", desired_sufix=desired_sufix)
    # initial_data = compute_sum_emisphres(data_logitudinal_tb, join_mean, sufix="dice", desired_sufix=desired_sufix) / 2
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    for i, approach in enumerate(approaches):
        if approach in [None, "initial"]:
            data = initial_data
        elif approach == "gt_registration":
            data = compute_mean_emisphres(data_registration_tb, join_mean, sufix="dice", desired_sufix=desired_sufix)
        else:
            data = compute_mean_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix)
            # data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix) / 2
        
        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=100, std=True)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.2f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_dice_table()
df_mean_hunt3, df_sem_hunt3, df_t_test_hunt3, df_p_value_hunt3 = display_dice_table(hunt=3)
# df_mean.to_csv(f"{current_path}/dice_mean.csv")

# new_columns = [
#     # "Metric", 
#     "lat. Vent.", 
#     "hippo.", 
#     "thala.", 
#     "ent. cortex", 
#     "inf. temp.", 
#     "sup. par.", 
#     "mid. fron.", 
#     "sup. fron.", 
#     "precuneus", 
#     "cuneus"
# ]
# df_mean = pd.DataFrame(df_mean.values, columns=new_columns, index=df_mean.index)
# df_sem = pd.DataFrame(df_sem.values, columns=new_columns, index=df_mean.index)
print("means HUNT4")
display(df_mean)
# print("means HUNT3")
# display(df_mean_hunt3)
print("std error")
display(df_sem)
print("t-test")
display(df_t_test)
print("p-value")
display(df_p_value)

# Volume precentage (increase/decrease)

In [ ]:
def get_vol_percentage(data_base, data, colums):
    _data = pd.DataFrame()
    for column in colums:
        _data[column] = data[column] / data_base[column]
    return _data

def get_vol_gt_registration():
    _c = [f"{ti}_vol" for ti in IMPORTANT_TISSUES]
    # display(data_cross_sectional_tb[[f"{__c}_hunt4" for __c in _c]].head())
    # display(data_registration_tb[[f"{__c}_diff" for __c in _c]].head())
    _data = data_cross_sectional_tb[[f"{__c}_hunt4" for __c in _c]].values - data_registration_tb[[f"{__c}_diff" for __c in _c]].values
    _data = pd.DataFrame(_data, columns=_c)
    return _data

def display_volumn_table():
    # approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "diffusion_model"]
    # approaches = ["initial", "gt_registration",  "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "diffusion_model"]
    approaches = ["expected", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="vol"
    data_hunt3 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt3", desired_sufix=desired_sufix)
    data_hunt4 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt4", desired_sufix=desired_sufix)
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    vox_plot = np.empty((0, len(val_images), len(columns)))

    initial_data = get_vol_percentage(data_hunt3, data_hunt4, [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean])
    for i, approach in enumerate(approaches):
        if approach in [None, "initial", "expected"]:
            data = initial_data
        elif approach == "gt_registration":
            # display(get_vol_gt_registration().head())
            _data = compute_sum_emisphres(get_vol_gt_registration(), join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = get_vol_percentage(data_hunt3, _data, columns)
        else:
            _data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = get_vol_percentage(data_hunt3, _data, columns)
            # print(data.min().values, data.max().values)

        vox_plot = np.vstack((vox_plot, data.values[np.newaxis, :, :])).astype('float32')
        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=1)

        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value, vox_plot

pd.options.display.float_format = '{:.5f}'.format

df_mean, df_sem, df_t_test, df_p_value, vox_plot = display_volumn_table()
# print(vox_plot[..., 0].shape)
# print(vox_plot[..., 0])
# vis.plot_data_voxplot(vox_plot[..., 1].T, ["expected", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"])

# df_mean.to_csv(f"{current_path}/dice_mean.csv")
# print("means HUNT4")
# display(df_mean)

# print("std error")
# display(df_sem)
# print("t-test")
# display(df_t_test)
# print("p-value")
# display(df_p_value)





# vol relative absolute volume difference

In [ ]:

def get_relative_absolute_volume_difference(data_base, data, colums):
    _data = pd.DataFrame()
    for column in colums:
        _data[column] = np.abs(data[column] - data_base[column]) / data_base[column]
    return _data

def get_absolute_symmetrized_percent_volume_change(data_base, data, colums):
    _data = pd.DataFrame()
    for column in colums:
        _data[column] = np.abs(data[column] - data_base[column]) / (0.5*(data_base[column]+data[column]))
    return _data

def compute_measure(data_base, data, colums, measure="ASPVC"):
    if measure == "ASPVC":
        return get_absolute_symmetrized_percent_volume_change(data_base, data, colums)
    else:
        return get_relative_absolute_volume_difference(data_base, data, colums)
    
def display_volumn_table(measure="ASPVC"):
    # approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "StarGAN_V2_sagital"]
    # approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "gt_registration"]
    approaches = ["initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"]
    # approaches = ["initial", "gt_registration",  "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "diffusion_model"]
    # approaches = ["initial", "gt_registration", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="vol"
    data_hunt3 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt3", desired_sufix=desired_sufix)
    data_hunt4 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt4", desired_sufix=desired_sufix)
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    initial_data = compute_measure(data_hunt4, data_hunt3, [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean], measure)
    for i, approach in enumerate(approaches):
        if approach in [None, "initial", "expected"]:
            data = initial_data
        elif approach == "gt_registration":
            _data = compute_sum_emisphres(get_vol_gt_registration(), join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = compute_measure(data_hunt4, _data, columns, measure)
        else:
            _data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = compute_measure(data_hunt4, _data, columns, measure)

        # data = (data - data.mean()) / data.std()

        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=100, std=True)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.3f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_volumn_table(measure="ASPVC")

# df_mean = pd.DataFrame(df_mean.values, columns=new_columns, index=df_mean.index)
# df_sem = pd.DataFrame(df_sem.values, columns=new_columns, index=df_mean.index)
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean)

print("std error")
display(df_sem)
print("t-test")
display(df_t_test)
print("p-value")
display(df_p_value)



# Volume Percentage

In [ ]:

def get_vol_percentage_df(data, sufix, exclude_tissues, total=None, ret_total=False):
    # ---- obtain just important df and tissues
    columnas_vol_hunt3 = data.filter(regex=f'{sufix}$', axis=1)
    tissues_ids = [int(i.split("_")[0]) for i in columnas_vol_hunt3.columns]
    tissues_ids = [i for i in tissues_ids if i not in exclude_tissues]
    reconstruc_columns = [f"{i}_{sufix}" for i in tissues_ids]
    _util_data = data[reconstruc_columns].copy()

    _data = pd.DataFrame()
    if total is None:
        total = _util_data[reconstruc_columns].sum(axis=1)
    for column in reconstruc_columns:
        _data[column] = _util_data[column] / total
    if ret_total:
        return _data, total
    else:
        return _data

def get_all_tissues_ids():
    columnas_vol_hunt3 = data_cross_sectional_tb.filter(regex=f'vol_hunt4$', axis=1)
    tissues_ids = [int(i.split("_")[0]) for i in columnas_vol_hunt3.columns]
    return tissues_ids

def get_all_vol_gt_registration():
    _c = [f"{ti}_vol" for ti in get_all_tissues_ids()]
    _data = data_cross_sectional_tb[[f"{__c}_hunt4" for __c in _c]].values - data_registration_tb[[f"{__c}_diff" for __c in _c]].values
    _data = pd.DataFrame(_data, columns=_c)
    return _data

def display_volumn_percentage_table():
    approaches = ["baseline", "follow_up", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "gt_registration"]
    # approaches = ["baseline", "follow_up", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="vol"
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    data_vol_hunt3, total = get_vol_percentage_df(data_cross_sectional_tb, 'vol_hunt3', [0], total=None, ret_total=True)
    data_hunt3 = compute_sum_emisphres(data_vol_hunt3, join_mean, sufix='vol_hunt3', desired_sufix=desired_sufix)

    data_vol_hunt4 = get_vol_percentage_df(data_cross_sectional_tb, 'vol_hunt4', [0], total=total)
    data_hunt4 = compute_sum_emisphres(data_vol_hunt4, join_mean, sufix='vol_hunt4', desired_sufix=desired_sufix)

    initial_data = data_hunt4
    for i, approach in enumerate(approaches):
        if approach == "baseline":
            data = data_hunt3
        elif approach == "follow_up":
            data = initial_data
        elif approach == "gt_registration":
            _data = get_vol_percentage_df(get_all_vol_gt_registration(), 'vol', [0], total=total)
            data = compute_sum_emisphres(_data, join_mean, sufix='vol', desired_sufix=desired_sufix)
        else:
            _data = get_vol_percentage_df(data_approaches_tb[approach], 'vol', [0], total=total)
            data = compute_sum_emisphres(_data, join_mean, sufix='vol', desired_sufix=desired_sufix)

        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, data_hunt3, columns, mult=100, std=True)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.3f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_volumn_percentage_table()

# df_mean = pd.DataFrame(df_mean.values, columns=new_columns, index=df_mean.index)
# df_sem = pd.DataFrame(df_sem.values, columns=new_columns, index=df_mean.index)
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean)

print("std error")
display(df_sem)
print("t-test")
display(df_t_test)
print("p-value")
display(df_p_value)



In [ ]:
pd.options.display.float_format = '{:.3f}'.format

data_vol_hunt3, total = get_vol_percentage_df(data_cross_sectional_tb, 'vol_hunt3', [0], total=None, ret_total=True)
data_hunt3 = compute_sum_emisphres(data_vol_hunt3, join_mean, sufix='vol_hunt3', desired_sufix="vol")

data_vol_hunt4 = get_vol_percentage_df(data_cross_sectional_tb, 'vol_hunt4', [0], total=total)
data_hunt4 = compute_sum_emisphres(data_vol_hunt4, join_mean, sufix='vol_hunt4', desired_sufix="vol")

_data_df = get_vol_percentage_df(data_approaches_tb["dfield_similar_patches"], 'vol', [0], total=total)
data_df = compute_sum_emisphres(_data_df, join_mean, sufix='vol', desired_sufix="vol")

_data_gan = get_vol_percentage_df(data_approaches_tb["MPGAN_paper_16_seg_loss"], 'vol', [0], total=total)
data_gan = compute_sum_emisphres(_data_gan, join_mean, sufix='vol', desired_sufix="vol")

_data_reg = get_vol_percentage_df(get_all_vol_gt_registration(), 'vol', [0], total=total)
data_reg = compute_sum_emisphres(_data_reg, join_mean, sufix='vol', desired_sufix="vol")

# display(data_cross_sectional_tb.head())
# display(data_vol_hunt3[["3_vol_hunt3", "42_vol_hunt3"]].head())
# display(data_hunt3.head())

# display(data_vol_hunt4[["3_vol_hunt4", "42_vol_hunt4"]].head())
# display(data_hunt4.head())

ddc = {"lat. ventricles": "43_4_...", "hippocampus": "53_17_...", "thalamus": "49_10_...", "cortex": "42_3_...", "entorhinal cortex": "2006_1006_...", "inf. temporal. gyri": "2009_1009_...", "sup. parietal": "2029_1029_...", "mid. frontal gyrus": "2027_1027_...", "sup. frontal gyrus": "2028_1028_...", "precuneus": "2025_1025_...", "cuneus": "2005_1005_..."}


def plot_volumen_percentage(data_hunt3, data_hunt4, tissue, figsize=(10, 6)):
    vols = {"CSF": "1_1_vol", "GM": "2_2_vol", "WM": "3_3_vol", "ventricles": "43_4_vol", "hipocampus":"53_17_vol", "thalamus":"49_10_vol",  "cortex":"42_3_vol", "entorhinal cortex":"2006_1006_vol", "inf. temporal. gyri":"2009_1009_vol", "sup. parietal":"2029_1029_vol", "mid. frontal gyrus":"2027_1027_vol", "sup. frontal gyrus":"2028_1028_vol", "precuneus":"2025_1025_vol", "cuneus":"2005_1005_vol"}
    title = f"{tissue} Volume Percentage"

    new_df = pd.DataFrame()
    new_df[f"{tissue}_hunt3"] = data_hunt3[vols[tissue]]
    new_df[f"{tissue}_hunt4"] = data_hunt4[vols[tissue]]
    new_df[f"{tissue}_DF"] = data_df[vols[tissue]]
    new_df[f"{tissue}_GAN"] = data_gan[vols[tissue]]
    new_df[f"{tissue}_REG"] = data_reg[vols[tissue]]

    new_df *= 100
    print(np.sum(new_df[f"{tissue}_hunt3"].values - new_df[f"{tissue}_hunt4"].values))
    df_melted = new_df.melt(var_name='HUNT', value_name='Percentage')
    plt.figure(figsize=figsize)
    sns.boxplot(x='HUNT', y='Percentage', data=df_melted, palette="pastel")
    sns.stripplot(x='HUNT', y='Percentage', data=df_melted, color='black', alpha=0.3, jitter=True)
    plt.title(title)

    plt.show()


tissue = "thalamus"
plot_volumen_percentage(data_hunt3, data_hunt4, tissue=tissue)

# ICV_1006

## brain parechimalfraction


In [ ]:
from sklearn.linear_model import LinearRegression

def calculate_slopes(df):
    # Convertir las categorías en valores numéricos
    df['category_num'] = df['category'].map({'Low': 0, 'Medium': 1, 'High': 2})

    methods = ['Baseline', 'Follow-up', 'Patches', 'MPGAN']
    slopes = {}

    for method in methods:
        X = df[['category_num']].values
        y = df[method].values
        model = LinearRegression()
        model.fit(X, y)
        slopes[method] = model.coef_[0]
    
    return slopes


def calculate_slopes_in_degrees(df):
    # Convertir las categorías en valores numéricos
    df['category_num'] = df['category'].map({'Low': 0, 'Medium': 1, 'High': 2})

    methods = ['Baseline', 'Follow-up', 'Patches', 'MPGAN']
    slopes_in_degrees = {}

    for method in methods:
        X = df[['category_num']].values
        y = df[method].values
        model = LinearRegression()
        model.fit(X, y)
        slope_radians = np.arctan(model.coef_[0])
        slope_degrees = np.degrees(slope_radians)
        slopes_in_degrees[method] = slope_degrees
    
    return slopes_in_degrees

In [ ]:

custom_palette = {
    'Baseline': np.array((192, 57, 43))/255,
    'Follow-up': np.array((142, 68, 173))/255,

    'Patches': np.array((41, 128, 185))/255,
    'MPGAN': np.array((40, 180, 99))/255,

    'MPGAN2': np.array((15, 200, 85))/255,
    # 'dice_1_HUNT3': np.array((86, 101, 115 ))/255,
    # 'dice_1_HUNT4': np.array((133, 146, 158 ))/255,
    # 'dice_2_HUNT3': np.array((41, 128, 185))/255,
    # 'dice_2_HUNT4': np.array((127, 179, 213 ))/255,
    # 'dice_3_HUNT3': np.array((22, 160, 133))/255,
    # 'dice_3_HUNT4': np.array((130, 224, 170))/255,
}

palete_list = ['Low Patches', 'High Patches', 'High MPGAN', 'Low Baseline', 'Low Follow-up', 'High Baseline', 'Medium Follow-up', 'High Follow-up', 'Low MPGAN', 'Medium Patches', 'Medium Baseline', 'Medium MPGAN', 'High MPGAN2', 'Low MPGAN2', 'Medium MPGAN2']
for key in palete_list:
    custom_palette[key] = custom_palette[key.split(" ")[-1]]


# custom_palette = "pastel"
def dislay_bpf_table(df_bpf, tytle="BPF", melted=False, figsize=(20, 6)):
    if not melted:
        df_melted = df_bpf.melt(var_name='Approach', value_name='BPF')
    else:
        df_melted = df_bpf
    # Crear el boxplot con jitter
    plt.figure(figsize=figsize)
    sns.boxplot(x='Approach', y='BPF', data=df_melted, palette=custom_palette)
    sns.stripplot(x='Approach', y='BPF', data=df_melted, color='black', alpha=0.3, jitter=True)
    plt.title(tytle)
    plt.savefig(get_save_image_path_name("BPF_low_mdeium_high"), dpi=600, bbox_inches='tight')
    # plt.savefig(get_save_image_path_name("BPF_all"), dpi=600, bbox_inches='tight')

    plt.show()

def read_bpf_table():
    # columns = ["hunt3", "hunt4", "gt_registration", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]
    # desired_columns = ["bpf_hunt3", "bpf_hunt4", "bpf_reg_gt", "bpf_patches", "bpf_MPGAN", "bpf_MGAN", "bpf_StarGAN"]

    columns = ["hunt3", "hunt4", "dfield_similar_patches", "MPGAN_paper_16_seg_loss"]
    desired_columns = ["Baseline", "Follow-up", "Patches", "MPGAN"]

    bpf_table = np.zeros((len(val_images), len(columns)))
    for i, column in enumerate(columns):
        if column in ["hunt3", "hunt4"]:
            data = data_cross_sectional_wbb[f"bpf_{column}"].values
        elif column == "gt_registration":
            data = data_registration_wbb["bpf_hunt3"].values
        else:
            data = data_approaches_wbb[column]["bpf"].values
        bpf_table[:, i] = data
    df_bpf = pd.DataFrame(bpf_table, columns=desired_columns)
    return df_bpf

def display_all_bpf():
    df_bpf = read_bpf_table()
    dislay_bpf_table(df_bpf, tytle="Brain Parenchymal Fraction (BPF)", figsize=(8, 4)) 

def display_category_bpf():
    df_bpf = read_bpf_table()

    approaches_columns = df_bpf.columns
    # Calcular los percentiles 33 y 66 para "bpf_hunt3"
    percentil_33 = np.percentile(df_bpf[df_bpf.columns[0]], 33)
    percentil_66 = np.percentile(df_bpf[df_bpf.columns[0]], 66)

    print(f"Percentil 33: {percentil_33}, Percentil 66: {percentil_66}")
    # Asignar categorías basado en "bpf_hunt3"
    def categorize_sample(sample):
        if sample <= percentil_33:
            return 'Low'
        elif sample <= percentil_66:
            return 'Medium'
        else:
            return 'High'
        
    df_bpf['category'] = df_bpf[df_bpf.columns[0]].apply(categorize_sample)
    print(calculate_slopes_in_degrees(df_bpf))
    # category_counts = df_bpf['category'].value_counts()
    category_means = df_bpf.groupby('category').mean()
    print(category_means)
    # print(df_bpf.head())
    # vis.plot_hits_df(df_bpf, column="Baseline", bins="auto", binrange=None, figsize=(8, 4), kde=False)
    # vis.plot_hits_df(df_bpf, column="Follow-up", bins="auto", binrange=None, figsize=(8, 4), kde=False)
    # vis.plot_hits_df(df_bpf, column="Patches", bins="auto", binrange=None, figsize=(8, 4), kde=False)
    # vis.plot_hits_df(df_bpf, column="MPGAN", bins="auto", binrange=None, figsize=(8, 4), kde=False)

    combined_data = []

    for index, row in df_bpf.iterrows():
        for approach_name in approaches_columns:
              combined_data.append((row['category'] + ' ' + approach_name, row[approach_name]))

    df_combined = pd.DataFrame(combined_data, columns=['Approach', 'BPF'])
    dislay_bpf_table(df_combined, tytle="Brain Parenchymal Fraction (BPF)", melted=True, figsize=(15, 6)) 

from matplotlib import rcParams

# Configurar Matplotlib para usar Times New Roman
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Times New Roman'] + rcParams['font.serif']

# display_all_bpf()s
display_category_bpf()

# Find best patient to show

In [ ]:
# data_logitudinal_wbb = data_logitudinal_wbb.reset_index()
def get_difference_by_wbb_metric(approach, metric="ssim"):
    org_data = data_logitudinal_wbb.reset_index()
    df_diff = pd.concat([data_approaches_wbb[approach][["patient_id", f"{metric}_hunt4"]], org_data[[metric, "bpf_hunt3", "bpf_hunt4"]]], axis=1)
    df_diff["diff"] = df_diff[f"{metric}_hunt4"] - df_diff[metric]
    return df_diff

approaches = ["dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital", "diffusion_model"]
df_diff = get_difference_by_wbb_metric(approaches[0], "psnr")
df_diff = df_diff.sort_values(by='diff', ascending=False)
display(df_diff.head(10))

# Percentil 33: 0.7393769325901478, Percentil 66: 0.7581957874899905

LOW = 78 #17112
MEDIUM = 80 #17310
HIGH = 48 #15170

# Compute individual patien statistics

In [ ]:
def display_patient_statistics(id, merge_emispheres=False):
    approaches = ["dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MGAN_paper_16", "StarGAN_V2_sagital"]
    wbb_columns_initial = ["ssim", "ncc", "psnr", "bpf_hunt3", "bpf_hunt4"]
    wbb_columns = ["ssim_hunt4", "ncc_hunt4", "psnr_hunt4", "bpf"]

    # -------- whole brain based statistics
    wbb_initial_metrics = data_logitudinal_wbb.loc[data_logitudinal_wbb['patient_id'] == int(id)][wbb_columns_initial].values
    wbb_approach_metrics = []
    for approach in approaches:
        wbb_data_patient = data_approaches_wbb[approach].loc[data_approaches_wbb[approach]['patient_id'] == int(id)][wbb_columns].values
        wbb_data_patient = np.insert(wbb_data_patient, 3, 0, axis=1)
        wbb_approach_metrics.append(wbb_data_patient)
 
    # df_patient = pd.DataFrame(np.concatenate([initial_metrics] + wbb_approach_metrics, axis=0), columns=wbb_columns_initial)
    # df_patient.insert(0, 'approach', ["initial"] + approaches)

    # -------- tissue based statistics
    # ---- dice
    tb_dice_columns_initial = [f"{_t}_dice" for _t in IMPORTANT_TISSUES] 
    tb_dice_columns = [f"{_t}_dice_hunt4" for _t in IMPORTANT_TISSUES] 
    tb_dice_desired_columns = [f"{_t}_dice" for _t in IMPORTANT_TISSUES]

    tb_dice_initial_metrics = data_logitudinal_tb.loc[data_logitudinal_tb['patient_id'] == int(id)][tb_dice_columns_initial]
    if merge_emispheres:
        tb_dice_initial_metrics = compute_mean_emisphres(tb_dice_initial_metrics, join_mean, sufix=f"dice", desired_sufix="dice") # you can comment this cde if you want separated dices
        tb_dice_desired_columns = [f"{t1}_{t2}_dice" for t1, t2 in join_mean]

    tb_dice_initial_metrics = tb_dice_initial_metrics.values
    tb_dice_approach_metrics = []
    for approach in approaches:
        tb_data_patient = data_approaches_tb[approach].loc[data_approaches_tb[approach]['patient_id'] == int(id)][tb_dice_columns]
        if merge_emispheres:
            tb_data_patient = compute_mean_emisphres(tb_data_patient, join_mean, sufix=f"dice_hunt4", desired_sufix="dice") # you can comment this cde if you want separated dices
        tb_dice_approach_metrics.append(tb_data_patient.values)

    # ---- vol diff
    tb_vol_columns_base = [f"{_t}_vol_hunt3" for _t in IMPORTANT_TISSUES]
    tb_vol_columns_initial = [f"{_t}_vol_hunt4" for _t in IMPORTANT_TISSUES]
    tb_vol_columns = [f"{_t}_vol" for _t in IMPORTANT_TISSUES]
    tb_vol_desired_columns = [f"{_t}_vol" for _t in IMPORTANT_TISSUES]

    tb_vol_hunt3 = data_cross_sectional_tb.loc[data_cross_sectional_tb['patient_id'] == int(id)][tb_vol_columns_base]
    tb_vol_hunt4 = data_cross_sectional_tb.loc[data_cross_sectional_tb['patient_id'] == int(id)][tb_vol_columns_initial]
    # display(tb_vol_hunt3)

    if merge_emispheres:
        tb_vol_hunt3 = compute_sum_emisphres(tb_vol_hunt3, join_mean, sufix="vol_hunt3", desired_sufix="vol")
        tb_vol_hunt4 = compute_sum_emisphres(tb_vol_hunt4, join_mean, sufix="vol_hunt4", desired_sufix="vol")
        tb_vol_desired_columns = [f"{t1}_{t2}_vol" for t1, t2 in join_mean]
    else:
        tb_vol_hunt3 = tb_vol_hunt3.rename(columns=dict([(f"{_t}_vol_hunt3", f"{_t}_vol") for _t in IMPORTANT_TISSUES]))
        tb_vol_hunt4 = tb_vol_hunt4.rename(columns=dict([(f"{_t}_vol_hunt4", f"{_t}_vol") for _t in IMPORTANT_TISSUES]))

        
    tb_vol_initial_metrics = get_relative_absolute_volume_difference(tb_vol_hunt4, tb_vol_hunt3, tb_vol_desired_columns)
    tb_vol_approach_metrics = []
    for approach in approaches:
        tb_data_patient = data_approaches_tb[approach].loc[data_approaches_tb[approach]['patient_id'] == int(id)][tb_vol_columns]
        if merge_emispheres:
            tb_data_patient = compute_sum_emisphres(tb_data_patient, join_mean, sufix=f"vol", desired_sufix="vol")
        tb_vol_approach_metrics.append(get_relative_absolute_volume_difference(tb_vol_hunt4, tb_data_patient, tb_vol_desired_columns).values)




    initial_metrics = np.concatenate([wbb_initial_metrics, tb_dice_initial_metrics, tb_vol_initial_metrics], axis=1)
    approach_metrics = [np.concatenate([wbb, tb_dice, tb_vol], axis=1) for wbb, tb_dice, tb_vol in zip(wbb_approach_metrics, tb_dice_approach_metrics, tb_vol_approach_metrics)]

    df_patient = pd.DataFrame(np.concatenate([initial_metrics] + approach_metrics, axis=0), columns=wbb_columns_initial + tb_dice_desired_columns + tb_vol_desired_columns)
    df_patient.insert(0, 'approach', ["initial"] + approaches)
    display(df_patient)

display_patient_statistics(val_images[LOW], merge_emispheres=True)
display_patient_statistics(val_images[MEDIUM], merge_emispheres=True)
display_patient_statistics(val_images[HIGH], merge_emispheres=True)

In [ ]:
columns_important_tissues_org = [f"{i}_dice" for i in IMPORTANT_TISSUES]
columns_important_tissues_gen = [f"{i}_dice_hunt4" for i in IMPORTANT_TISSUES]
columns_important_tissues_div = [f"{i}_div" for i in IMPORTANT_TISSUES]

df_comparation = data_logitudinal_tb[["patient_id"]].copy()
df_comparation[columns_important_tissues_org] = data_logitudinal_tb[columns_important_tissues_org]
df_comparation[columns_important_tissues_gen] = data_approaches_tb["dfield_similar_patches"][columns_important_tissues_gen]

df_comparation[columns_important_tissues_div] = data_logitudinal_tb[columns_important_tissues_org].values / data_approaches_tb["dfield_similar_patches"][columns_important_tissues_gen].values
# mask = df_comparation[columns_important_tissues_div] >= 1
# print(mask.head(30))
# count_ge_1 = mask.sum()
# print(count_ge_1)

# display(df_comparation.head(2))
# display(df_comparation[["patient_id"] + columns_div].head(20))
print("BPF MEAN:\n", df_comparation[columns_important_tissues_gen].mean()[1:])
print("\nBPF MAX:\n", df_comparation.max()[1:])

# display(data_logitudinal_tb)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Crear datos de ejemplo
np.random.seed(10)
data = pd.DataFrame({
    'category': np.repeat(['A', 'B', 'C', 'D'], 10),
    'value': np.concatenate([np.random.normal(10, 5, 10),
                             np.random.normal(15, 5, 10),
                             np.random.normal(25, 5, 10),
                             np.random.normal(5, 2, 10)])
})
display(data.head(100))
# Crear el boxplot con jitter
plt.figure(figsize=(10, 6))
sns.boxplot(x='category', y='value', data=data, palette="pastel")
sns.stripplot(x='category', y='value', data=data, color='black', alpha=0.3, jitter=True)
plt.title('A boxplot with jitter')
plt.show()
